# CXR Diffusion Demo on Kaggle

Self-contained launcher for the local `demo/app.py` (repository commit
`4cb8d14280fe780ecda25c8bffea64f99eedea62`). Model weights come from the public
[SillySom repository](https://huggingface.co/SillySom/ie643-cxr-diffusion-weights).
No Hugging Face token is needed.

1. In Kaggle, use **File > Import Notebook** to upload `cxr_demo_kaggle.ipynb`
   from your computer into a new **private** notebook. No other files are needed.
2. Under **Settings / Session options**, enable **Internet** and an available **GPU**.
   Complete Kaggle account verification yourself if prompted.
3. Run the three code cells in order. First setup installs packages and downloads models.
4. Open the printed `gradio.live` link. Use username **cxr** and the randomly generated
   password printed in the final cell. Both the password and link change on each launch.
5. Keep the final cell running while using the app. Interrupt the cell and select
   **Stop session** when finished to release the GPU.

If GPU options are disabled, check **Phone Verification** in your Kaggle account
settings, reload the notebook editor after verification, and check remaining GPU
quota. GPU access is an account/session setting; Python code cannot enable it.

The app uses one GPU, automatic bf16/fp16 selection, VAE slicing, and a
default of one image per generator. fp16 results can differ from the bf16 report.
The generated link routes through Gradio and requires a session password.
Do not upload sensitive patient data. Do not publish notebook outputs containing
the session password. A Kaggle runtime or Gradio tunnel failure ends the demo.

Research/coursework only. Tested on Kaggle with Tesla T4, Python 3.13, PyTorch 2.11.0+cu130, and CUDA 13.0; baseline and Margin + Adversarial generation and classification completed at seed 42, 50 steps, and CFG 7.5.


## 1. Check GPU, Internet, and public weights

In [ ]:
import json, subprocess, urllib.request
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle Settings / Session options, then restart this session.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA PyTorch:", torch.__version__, "CUDA", torch.version.cuda)
subprocess.run(["nvidia-smi"], check=True)

repo = "SillySom/ie643-cxr-diffusion-weights"
try:
    with urllib.request.urlopen("https://huggingface.co/api/models/" + repo, timeout=30) as response:
        info = json.load(response)
except Exception as exc:
    raise RuntimeError("Enable Internet in Kaggle and check that the Hugging Face weights repository is public.") from exc
if info.get("private") or info.get("gated"):
    raise RuntimeError("This notebook expects public, ungated model weights.")
files = {item["rfilename"] for item in info.get("siblings", [])}
required = {"novelty/prajiv_base/model_index.json", "novelty/heads/eval_head.pt",
            "novelty/heads/ood_stats.pt", "novelty/heads/anatomy.json", "novelty/heads/heads_report.json",
            "novelty/downstream/demo_real_only_seed42/best_model.pt",
            "novelty/downstream/demo_ce_adv_recipe_seed42/best_model.pt"}
missing = required - files
if missing:
    raise RuntimeError("Weights repository is missing files: " + ", ".join(sorted(missing)))
print("Public model repository is accessible without a token.")


## 2. Install the demo in an isolated environment

In [ ]:
import os, subprocess, sys, venv
from pathlib import Path

work = Path("/kaggle/working/cxr-demo")
work.mkdir(parents=True, exist_ok=True)
env_dir = work / ".venv"
# Kaggle may not provide a working ensurepip. This also repairs a partial venv.
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(env_dir)
demo_python = str(env_dir / "bin/python")
(work / 'app_kaggle.py').write_text('"""Standalone demo: text prompt -> synthetic chest X-ray -> classification + structural-defect check.\n\nSelf-contained: needs only this file and the packages in requirements.txt. The models (about 2.2 GB) are\ndownloaded from the Hugging Face repo on first run and cached (~/.cache/huggingface, or --cache_dir).\n\n  export HF_TOKEN=hf_...                     # optional token; weights are public\n  python app.py                              # -> http://127.0.0.1:7860\n  python app.py --local_root /path/to/ie643-cxr-diffusion   # use a full clone instead of downloading\n\nModels (exactly the checkpoints evaluated in the report):\n  * generators: phase-1 P-Rajiv LoRA (epoch_2) and every novelty LoRA at the step chosen by held-out checkpoint\n    selection; sampler = DDIM 50 steps, CFG 7.5 (report settings, adjustable in the UI)\n  * held-out judge: TorchXRayVision densenet121-res224-mimic_ch + 6-class linear probe; Mahalanobis OOD detector;\n    PSPNet lung/heart anatomy check\n  * downstream 14-class DenseNet-121: real-only and real + CE+Adv synthetic [recipe] (seed 42, validation-selected)\n"""\nimport os, json, argparse, threading\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom PIL import Image\n\n# This demo uses ordinary LoRA layers, not optional torchao quantization.\nimport importlib as _importlib\n_peft_utils = _importlib.import_module("peft.import_utils")\ntry:\n    _peft_utils.is_torchao_available()\nexcept ImportError as _error:\n    if "incompatible version of torchao" not in str(_error):\n        raise\n    _peft_tao = _importlib.import_module("peft.tuners.lora.torchao")\n    _peft_tao.is_torchao_available = lambda: False\n    print("[demo] Disabled unused incompatible torchao integration.", flush=True)\n\nDEMO_DTYPE = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16)\n\nHF_REPO = "SillySom/ie643-cxr-diffusion-weights"\n\n# ---------------------------------------------------------------------------------------------- what to load\nCLASSES = ["No Finding", "Enlarged Cardiomediastinum", "Lung Lesion", "Pneumonia", "Pleural Other", "Fracture"]\nDS_CLASSES = ["No Finding", "Lung Opacity", "Support Devices", "Pleural Effusion", "Edema", "Pneumothorax",\n              "Atelectasis", "Cardiomegaly", "Fracture", "Consolidation", "Enlarged Cardiomediastinum",\n              "Lung Lesion", "Pneumonia", "Pleural Other"]\nEVAL_BACKBONE = "densenet121-res224-mimic_ch"      # held-out judge backbone (MIMIC only, never used in training)\n\nBASELINE_LORA = "ie643_code/ie643/cxr_diffusion_checkpoints/prajiv/unet_clip/epoch_2"\n# key, UI label, LoRA dir (selected step = novelty/runs/<run>/best.json), report numbers:\n# requested finding %, defects %, KID x1e3, downstream macro-F1 (plain augmentation, mean of 3 seeds)\nGENERATORS = [\n    ("baseline", "P-Rajiv LoRA (baseline, phase 1)", BASELINE_LORA, 15.7, 32.7, 12.8, 0.256),\n    ("control", "Control (+2k steps, no new loss)", "novelty/runs/control/step_2000", 23.6, 51.8, 10.3, 0.275),\n    ("ce_only", "CE only (reward-hacks)", "novelty/runs/ce_only/step_1000", 35.3, 78.5, 29.9, 0.258),\n    ("adv_only", "Adversarial only", "novelty/runs/adv_only/step_2000", 25.9, 9.8, 2.9, 0.270),\n    ("ce_adv", "CE + adversarial", "novelty/runs/ce_adv/step_1000", 45.9, 24.6, 8.2, 0.269),\n    ("margin_adv", "Margin + adversarial (best)", "novelty/runs/margin_adv/step_1500", 51.2, 18.2, 7.8, 0.278),\n    ("margin_adv_ms", "Margin + adv, multi-scale D", "novelty/runs/margin_adv_ms/step_1000", 43.8, 16.5, 8.7, 0.264),\n]\nLABEL2KEY = {g[1]: g[0] for g in GENERATORS}\nKEY2LABEL = {g[0]: g[1] for g in GENERATORS}\nCLASSIFIERS = [  # key, UI label, folder\n    ("real", "DenseNet-121, real only", "novelty/downstream/demo_real_only_seed42"),\n    ("syn", "DenseNet-121, real + CE+Adv synthetic [recipe]", "novelty/downstream/demo_ce_adv_recipe_seed42"),\n]\nHEADS = "novelty/heads"\nBASE = "novelty/prajiv_base"\nTEST_CSV = "ie643_code/ie643/global_test_single_frontal_clean.csv"\n\nALLOW = ([f"{BASE}/*", f"{HEADS}/eval_head.pt", f"{HEADS}/ood_stats.pt", f"{HEADS}/anatomy.json",\n          f"{HEADS}/heads_report.json", TEST_CSV]\n         + [f"{g[2]}/unet_lora/*" for g in GENERATORS] + [f"{g[2]}/text_encoder_lora/*" for g in GENERATORS]\n         + [f"{c[2]}/best_model.pt" for c in CLASSIFIERS] + [f"{c[2]}/log.json" for c in CLASSIFIERS])\n\n\ndef fetch_assets(repo, cache_dir=None, token=None):\n    from huggingface_hub import snapshot_download\n    print(f"[demo] fetching models from https://huggingface.co/{repo} (first run downloads ~2.2 GB) ...", flush=True)\n    return snapshot_download(repo, allow_patterns=ALLOW, cache_dir=cache_dir, token=token)\n\n\ndef prompt_for(cls):\n    return "Normal chest X-ray with no significant findings" if cls == "No Finding" else "Chest X-ray showing " + cls\n\n\ndef requested_class(prompt):\n    p = prompt.lower()\n    for c in CLASSES[1:]:\n        if c.lower() in p:\n            return c\n    if "no finding" in p or "normal" in p:\n        return "No Finding"\n    return None\n\n\n# ---------------------------------------------------------------------------------------------- model pieces\n# (identical to novelty/cxr_common.py and novelty/sd_utils.py, so outputs match the report)\ndef to_xrv(img, size=224):\n    """(B,3|1,H,W) in [-1,1] -> (B,1,size,size) in [-1024,1024] (TorchXRayVision input range)."""\n    if img.shape[1] == 3:\n        img = img.mean(1, keepdim=True)\n    if img.shape[-1] != size:\n        img = F.interpolate(img.float(), size=(size, size), mode="bilinear", align_corners=False, antialias=True)\n    return img.float().clamp(-1, 1) * 1024.0\n\n\ndef backbone_features(m, x_xrv):\n    return F.adaptive_avg_pool2d(F.relu(m.features(x_xrv)), 1).flatten(1)\n\n\nclass LinearHead(nn.Module):\n    def __init__(self, dim=1024, n=len(CLASSES)):\n        super().__init__()\n        self.register_buffer("mu", torch.zeros(dim))\n        self.register_buffer("sd", torch.ones(dim))\n        self.fc = nn.Linear(dim, n)\n\n    def forward(self, f):\n        return self.fc((f - self.mu) / self.sd)\n\n\nclass AnatomyChecker:\n    """TorchXRayVision PSPNet lungs/heart; passes iff one dominant left and right lung, plausible areas, correct sides."""\n    def __init__(self, device, th):\n        import torchxrayvision as xrv\n        self.seg = xrv.baseline_models.chestx_det.PSPNet().to(device).eval()\n        t = self.seg.targets\n        self.iL, self.iR, self.iH = t.index("Left Lung"), t.index("Right Lung"), t.index("Heart")\n        self.device, self.th = device, th\n\n    @torch.no_grad()\n    def masks(self, img):\n        return (torch.sigmoid(self.seg(to_xrv(img.to(self.device), 512))) > 0.5).cpu().numpy()\n\n    def measure(self, m):\n        from scipy import ndimage\n        out = {k: [] for k in ["areaL", "areaR", "compL", "compR", "heart_area", "cxL", "cxR", "cxH"]}\n        for b in range(m.shape[0]):\n            for key, idx in (("L", self.iL), ("R", self.iR)):\n                mask = m[b, idx]\n                lab, n = ndimage.label(mask)\n                sizes = np.bincount(lab.ravel())[1:] if n else np.array([0])\n                out["area" + key].append(mask.mean())\n                out["comp" + key].append(int((sizes > 0.02 * mask.size).sum()))\n                xs = np.nonzero(mask)[1]\n                out["cx" + key].append(xs.mean() / mask.shape[1] if len(xs) else np.nan)\n            h = m[b, self.iH]\n            out["heart_area"].append(h.mean())\n            xs = np.nonzero(h)[1]\n            out["cxH"].append(xs.mean() / h.shape[1] if len(xs) else np.nan)\n        return {k: np.array(v, dtype=float) for k, v in out.items()}\n\n    def passes(self, meas):\n        th, ok = self.th, np.ones(len(meas["areaL"]), dtype=bool)\n        for k in ("areaL", "areaR", "heart_area"):\n            ok &= (meas[k] >= th[k][0]) & (meas[k] <= th[k][1])\n        ok &= (meas["compL"] == 1) & (meas["compR"] == 1)\n        ok &= np.nan_to_num(meas["cxR"], nan=1.0) < np.nan_to_num(meas["cxL"], nan=0.0)   # patient\'s right = image left\n        ok &= (np.nan_to_num(meas["cxH"], nan=-1) > np.nan_to_num(meas["cxR"], nan=2)) & \\\n              (np.nan_to_num(meas["cxH"], nan=2) < np.nan_to_num(meas["cxL"], nan=-1))\n        return ok\n\n\ndef encode_prompts(pipe, prompts, device):\n    ids = pipe.tokenizer(prompts, padding="max_length", max_length=pipe.tokenizer.model_max_length,\n                         truncation=True, return_tensors="pt").input_ids.to(device)\n    return pipe.text_encoder(ids)[0]\n\n\n@torch.no_grad()\ndef sample(pipe, ddim, prompts, seeds, steps, cfg, device):\n    """Deterministic DDIM + classifier-free guidance; one CPU generator per seed (batch-independent noise)."""\n    with torch.autocast("cuda", dtype=DEMO_DTYPE):\n        cond, uncond = encode_prompts(pipe, prompts, device), encode_prompts(pipe, [""] * len(prompts), device)\n        z = torch.stack([torch.randn(4, 64, 64, generator=torch.Generator().manual_seed(int(s))) for s in seeds]\n                        ).to(device) * ddim.init_noise_sigma\n        ddim.set_timesteps(steps, device=device)\n        for t in ddim.timesteps:\n            out = pipe.unet(torch.cat([z, z]), t, encoder_hidden_states=torch.cat([uncond, cond])).sample\n            e_u, e_c = out.chunk(2)\n            z = ddim.step((e_u + cfg * (e_c - e_u)).float(), t, z.float()).prev_sample\n        z = z / pipe.vae.config.scaling_factor\n        return torch.cat([pipe.vae.decode(z[i:i + 4]).sample.float().clamp(-1, 1) for i in range(0, len(z), 4)])\n\n\n# ---------------------------------------------------------------------------------------------- engine\nclass Engine:\n    def __init__(self, root, device="cuda", data_dir=None):\n        from diffusers import StableDiffusionPipeline, DDIMScheduler\n        from peft import PeftModel\n        import torchxrayvision as xrv\n        from torchvision import transforms\n        P = lambda *a: os.path.join(root, *a)\n        self.root, self.dev, self.data_dir = root, device, data_dir\n        self.lock = threading.Lock()\n\n        print("[demo] loading P-Rajiv SD-1.5 + 7 LoRA adapters ...", flush=True)\n        self.pipe = StableDiffusionPipeline.from_pretrained(P(BASE), torch_dtype=DEMO_DTYPE, safety_checker=None,\n                                                            requires_safety_checker=False)\n        self.ddim = DDIMScheduler.from_config(self.pipe.scheduler.config)\n        for i, (key, _, d, *_r) in enumerate(GENERATORS):\n            u, t = P(d, "unet_lora"), P(d, "text_encoder_lora")\n            if i == 0:\n                self.pipe.unet = PeftModel.from_pretrained(self.pipe.unet, u, adapter_name=key)\n                self.pipe.text_encoder = PeftModel.from_pretrained(self.pipe.text_encoder, t, adapter_name=key)\n            else:\n                self.pipe.unet.load_adapter(u, adapter_name=key)\n                self.pipe.text_encoder.load_adapter(t, adapter_name=key)\n        for m in (self.pipe.unet, self.pipe.text_encoder, self.pipe.vae):\n            m.eval()\n            for n, prm in m.named_parameters():\n                if "lora_" in n:\n                    prm.data = prm.data.float()\n                prm.requires_grad_(False)\n        self.pipe.to(device)\n\n        print("[demo] loading held-out judge, OOD detector, anatomy checker ...", flush=True)\n        self.bb = xrv.models.DenseNet(weights=EVAL_BACKBONE).to(device).eval()\n        self.head = LinearHead().to(device)\n        self.head.load_state_dict(torch.load(P(HEADS, "eval_head.pt"), map_location=device))\n        self.head.eval()\n        o = torch.load(P(HEADS, "ood_stats.pt"), map_location="cpu")\n        self.ood_mean, self.ood_prec = o["mean"].float().to(device), o["prec"].float().to(device)\n        self.ood_thr = float(o["thr"])\n        self.anat = AnatomyChecker(device, json.load(open(P(HEADS, "anatomy.json"))))\n        self.judge_real_acc = json.load(open(P(HEADS, "heads_report.json")))["eval"]["test_acc"]\n\n        print("[demo] loading downstream DenseNet-121 classifiers ...", flush=True)\n        from torchvision import models\n        self.ds, self.ds_info = {}, {}\n        for key, _, d in CLASSIFIERS:\n            ck = P(d, "best_model.pt")\n            if not os.path.exists(ck):\n                self.ds_info[key] = "not available"\n                continue\n            m = models.densenet121(weights=None)\n            m.classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(m.classifier.in_features, 14))\n            sd = torch.load(ck, map_location="cpu")\n            m.load_state_dict(sd["state_dict"])\n            self.ds[key] = m.to(device).eval()\n            self.ds_info[key] = f"epoch {sd[\'epoch\']} selected on validation, test macro-F1 {sd[\'test_f1\']:.3f}"\n        self.ds_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),\n                                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])\n        self.test_df = None\n        if data_dir and os.path.isdir(os.path.join(data_dir, "CheXpert-v1.0-small")) and os.path.exists(P(TEST_CSV)):\n            self.test_df = pd.read_csv(P(TEST_CSV))\n        print("[demo] ready.", flush=True)\n\n    @torch.no_grad()\n    def generate(self, key, prompt, seeds, steps, cfg):\n        for net in (self.pipe.unet, self.pipe.text_encoder):\n            net.set_adapter(key)\n        return sample(self.pipe, self.ddim, [prompt] * len(seeds), seeds, int(steps), float(cfg), self.dev)\n\n    @torch.no_grad()\n    def analyse(self, x, pil_images, requested=None):\n        x = x.to(self.dev).float()\n        f = backbone_features(self.bb, to_xrv(x))\n        probs = torch.softmax(self.head(f), 1).cpu().numpy()\n        d = f - self.ood_mean\n        maha = torch.sqrt(((d @ self.ood_prec) * d).sum(1)).cpu().numpy()\n        masks = self.anat.masks(x)\n        meas = self.anat.measure(masks)\n        ok = self.anat.passes(meas)\n        ds_out = {}\n        if self.ds:\n            inp = torch.stack([self.ds_tf(im.convert("RGB")) for im in pil_images]).to(self.dev)\n            ds_out = {k: torch.softmax(m(inp), 1).cpu().numpy() for k, m in self.ds.items()}\n        out = []\n        for b in range(len(x)):\n            r = {"judge": {c: float(probs[b, i]) for i, c in enumerate(CLASSES)}, "maha": float(maha[b]),\n                 "ood": bool(maha[b] > self.ood_thr), "anatomy_ok": bool(ok[b]),\n                 "meas": {k: float(v[b]) for k, v in meas.items()}, "requested": requested}\n            r["defect"] = (not r["anatomy_ok"]) or r["ood"]\n            r["judge_top"] = max(r["judge"], key=r["judge"].get)\n            r["shows_requested"] = None if requested is None else (r["judge_top"] == requested)\n            for k in ds_out:\n                r["ds_" + k] = {c: float(ds_out[k][b, i]) for i, c in enumerate(DS_CLASSES)}\n            r["overlay"] = self.overlay(pil_images[b], masks[b])\n            out.append(r)\n        return out\n\n    def overlay(self, pil, m):\n        rgb = np.stack([np.asarray(pil.convert("L").resize((512, 512)), dtype=np.float32)] * 3, -1)\n        t = self.anat.seg.targets\n        for name, col in (("Left Lung", (60, 130, 255)), ("Right Lung", (40, 200, 120)), ("Heart", (240, 70, 70))):\n            mk = m[t.index(name)].astype(bool)\n            rgb[mk] = 0.6 * rgb[mk] + 0.4 * np.array(col, dtype=np.float32)\n        return Image.fromarray(rgb.clip(0, 255).astype(np.uint8))\n\n\ndef to_tensor_512(pil):\n    """Same preprocessing as training/evaluation: gray -> 512 bicubic -> [-1,1], 3 channels."""\n    g = np.asarray(pil.convert("L").resize((512, 512), Image.BICUBIC), dtype=np.float32) / 127.5 - 1.0\n    return torch.from_numpy(g)[None].repeat(3, 1, 1)\n\n\ndef to_pil(x):\n    return [Image.fromarray(((im.mean(0) + 1) * 127.5).round().clamp(0, 255).byte().cpu().numpy(), "L") for im in x]\n\n\n# ---------------------------------------------------------------------------------------------- UI\ndef build_ui(E):\n    import gradio as gr\n\n    def fmt_p(d):\n        c = max(d, key=d.get)\n        return f"{c} ({100 * d[c]:.0f}%)"\n\n    def summary_rows(results):\n        rows = []\n        for i, r in enumerate(results):\n            row = {"#": i + 1}\n            if "gen" in r:\n                row.update({"Generator": r["gen"], "Seed": r["seed"]})\n            row["Held-out judge: top class"] = fmt_p(r["judge"])\n            if r["requested"]:\n                row["p(requested)"] = f"{100 * r[\'judge\'][r[\'requested\']]:.0f}%"\n                row["Shows requested finding?"] = "yes" if r["shows_requested"] else "no"\n            row["Anatomy check"] = "pass" if r["anatomy_ok"] else "FAIL"\n            row["OOD"] = "yes" if r["ood"] else "no"\n            row["Structural defect?"] = "YES" if r["defect"] else "no"\n            for k, lab, _ in CLASSIFIERS:\n                row[lab] = fmt_p(r["ds_" + k]) if "ds_" + k in r else "(not available)"\n            rows.append(row)\n        return pd.DataFrame(rows)\n\n    def detail(r):\n        req = r["requested"]\n        lines = [f"**Requested finding:** {req or \'not one of the 6 trained classes\'}"\n                 + ("" if req is None else f" — held-out judge {\'agrees\' if r[\'shows_requested\'] else \'disagrees\'}"),\n                 f"**Anatomy check:** {\'pass\' if r[\'anatomy_ok\'] else \'FAIL\'} — lung components L/R = "\n                 f"{int(r[\'meas\'][\'compL\'])}/{int(r[\'meas\'][\'compR\'])} (need 1/1), lung area L/R = "\n                 f"{100 * r[\'meas\'][\'areaL\']:.1f}% / {100 * r[\'meas\'][\'areaR\']:.1f}%, heart area "\n                 f"{100 * r[\'meas\'][\'heart_area\']:.1f}%",\n                 f"**OOD (Mahalanobis, held-out features):** {r[\'maha\']:.1f} vs threshold {E.ood_thr:.1f} → "\n                 f"{\'out of distribution\' if r[\'ood\'] else \'in distribution\'}",\n                 f"**Structural defect:** {\'YES\' if r[\'defect\'] else \'no\'}  (= anatomy fail OR OOD)"]\n        return [r["overlay"], r["judge"], "\\n\\n".join(lines)] + [r.get("ds_" + k) for k, _, _ in CLASSIFIERS]\n\n    def run_generate(prompt, gens, n, seed, steps, cfg, progress=gr.Progress()):\n        if not prompt.strip():\n            raise gr.Error("Enter a prompt.")\n        if not gens:\n            raise gr.Error("Pick at least one generator.")\n        req = requested_class(prompt)\n        seeds = [int(seed) + i for i in range(int(n))]\n        results, gallery = [], []\n        with E.lock:\n            for gi, lab in enumerate(gens):\n                progress((gi, len(gens)), desc=f"Generating with {lab}")\n                x = E.generate(LABEL2KEY[lab], prompt, seeds, steps, cfg)\n                pil = to_pil(x)\n                for r, s, im in zip(E.analyse(x, pil, req), seeds, pil):\n                    r.update(gen=lab, seed=s)\n                    results.append(r)\n                    shows = "" if req is None else (" · shows finding" if r["shows_requested"] else " · finding not detected")\n                    gallery.append((im, f"#{len(results)} {lab} · seed {s} · {\'defect\' if r[\'defect\'] else \'ok\'}{shows}"))\n        return [gallery, summary_rows(results), results] + detail(results[0])\n\n    def on_select(results, evt: gr.SelectData):\n        return detail(results[evt.index]) if results else [gr.update()] * (3 + len(CLASSIFIERS))\n\n    def random_test(cls):\n        rows = E.test_df[E.test_df[cls] == 1.0]\n        row = rows.sample(1).iloc[0]\n        return Image.open(os.path.join(E.data_dir, row.Path)).convert("L"), cls\n\n    def run_classify(img, true_label):\n        if img is None:\n            raise gr.Error("Upload an image first.")\n        pil = img.convert("L")\n        with E.lock:\n            r = E.analyse(to_tensor_512(pil)[None], [pil])[0]\n        df = summary_rows([r])\n        if true_label:\n            df.insert(1, "True label (CheXpert)", true_label)\n        return [df] + detail(r)\n\n    gen_rows = "\\n".join(f"| {lab} | `{d}` | {rf}% | {df_}% | {kid} | {f1:.3f} |"\n                         for _, lab, d, rf, df_, kid, f1 in GENERATORS)\n    cls_rows = "\\n".join(f"- **{lab}**: {E.ds_info[k]}" for k, lab, _ in CLASSIFIERS)\n    about = f"""\n### Checkpoints (the ones evaluated in the report)\n| Generator | LoRA checkpoint | Requested finding ↑ | Defects ↓ | KID×10³ ↓ | Downstream macro-F1 |\n|---|---|---|---|---|---|\n{gen_rows}\n| *Real images (reference)* | n/a | 29.7% (judge accuracy) | n/a | n/a | **0.285** (real only) |\n\nReport numbers: mean over the 5 rare pathologies, 2,173 images per generator, 50 DDIM steps, CFG 7.5.\nDownstream: 14-class DenseNet-121, mean of 3 seeds, 700 real test images.\n\n### Classifiers in this demo\n{cls_rows}\n\n### How images are judged\n- **Held-out judge**: TorchXRayVision DenseNet-121 trained on MIMIC only (`mimic_ch`) + 6-class linear probe; never\n  used in training. On *real* test images it is **{100 * E.judge_real_acc:.1f}%** accurate (chance 16.7%), so a single\n  verdict is noisy. The report averages over hundreds of images.\n- **Structural defect** = anatomy check fails (PSPNet: exactly one left and one right lung, plausible areas, correct\n  sides) **or** Mahalanobis out-of-distribution in the judge\'s feature space.\n- Generators were trained on the 6 template prompts; free text works but is outside the training distribution.\n\nModels: https://huggingface.co/{HF_REPO}\n"""\n\n    with gr.Blocks(title="CXR Diffusion Demo") as demo:\n        gr.Markdown("# Text-Conditioned Synthetic Chest X-ray Generation\\n"\n                    "Gradient Mavericks · IE643. Generate chest X-rays from a prompt with the report\'s LoRA "\n                    "checkpoints, then see what the held-out judge, the anatomy/OOD defect check and the downstream "\n                    "DenseNet classifiers say about each image.")\n        results_state = gr.State([])\n        with gr.Tab("Generate & classify"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    preset = gr.Dropdown([prompt_for(c) for c in CLASSES], value=prompt_for("Pneumonia"),\n                                         label="Prompt preset (training prompts)")\n                    prompt = gr.Textbox(value=prompt_for("Pneumonia"), label="Prompt", lines=2)\n                    gens = gr.CheckboxGroup([g[1] for g in GENERATORS],\n                                            value=[KEY2LABEL["baseline"], KEY2LABEL["margin_adv"]],\n                                            label="Generators (same seeds for each → fair comparison)")\n                    with gr.Row():\n                        n = gr.Slider(1, 2, value=1, step=1, label="Images per generator")\n                        seed = gr.Number(value=42, precision=0, label="Seed")\n                    with gr.Accordion("Sampler (report: 50 steps, CFG 7.5)", open=False):\n                        steps = gr.Slider(10, 50, value=50, step=1, label="DDIM steps")\n                        cfg = gr.Slider(1.0, 12.0, value=7.5, step=0.5, label="Classifier-free guidance")\n                    go = gr.Button("Generate and classify", variant="primary")\n                with gr.Column(scale=2):\n                    gallery = gr.Gallery(label="Generated X-rays (click one for details)", columns=4, height=520,\n                                         object_fit="contain")\n            table = gr.Dataframe(label="Per-image verdicts", interactive=False, wrap=True)\n            with gr.Row():\n                ov = gr.Image(label="Anatomy segmentation (blue/green = lungs, red = heart)", height=380)\n                judge = gr.Label(label="Held-out judge (6 classes)", num_top_classes=6)\n                with gr.Column():\n                    ds_labels = [gr.Label(label=lab, num_top_classes=5) for _, lab, _ in CLASSIFIERS]\n            info = gr.Markdown()\n            preset.change(lambda p: p, preset, prompt)\n            go.click(run_generate, [prompt, gens, n, seed, steps, cfg],\n                     [gallery, table, results_state, ov, judge, info] + ds_labels)\n            gallery.select(on_select, results_state, [ov, judge, info] + ds_labels)\n\n        with gr.Tab("Classify an X-ray"):\n            gr.Markdown("Upload any frontal chest X-ray and run the same judge, defect check and classifiers."\n                        + ("" if E.test_df is not None else\n                           " *(Start the app with `--data_dir` pointing at a folder containing CheXpert-v1.0-small "\n                           "to also load random real test images.)*"))\n            with gr.Row():\n                with gr.Column(scale=1):\n                    up = gr.Image(type="pil", label="Chest X-ray", height=380)\n                    with gr.Row(visible=E.test_df is not None):\n                        cls_pick = gr.Dropdown(DS_CLASSES, value="Pneumonia", label="Test-set class")\n                        rnd = gr.Button("Load random real test image")\n                    true_lab = gr.Textbox(label="True label (only for test-set images)", interactive=False)\n                    cbtn = gr.Button("Classify", variant="primary")\n                with gr.Column(scale=2):\n                    ctable = gr.Dataframe(label="Verdict", interactive=False, wrap=True)\n                    with gr.Row():\n                        cov = gr.Image(label="Anatomy segmentation", height=360)\n                        cjudge = gr.Label(label="Held-out judge (6 classes)", num_top_classes=6)\n                    cds = [gr.Label(label=lab, num_top_classes=5) for _, lab, _ in CLASSIFIERS]\n                    cinfo = gr.Markdown()\n            up.upload(lambda: "", None, true_lab)\n            if E.test_df is not None:\n                rnd.click(random_test, cls_pick, [up, true_lab])\n            cbtn.click(run_classify, [up, true_lab], [ctable, cov, cjudge, cinfo] + cds)\n\n        with gr.Tab("About the models"):\n            gr.Markdown(about)\n    return demo\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--hf_repo", default=HF_REPO, help="Hugging Face repo with the weights")\n    ap.add_argument("--local_root", default=None, help="use a local clone of the code repo (+ downloaded weights) instead")\n    ap.add_argument("--cache_dir", default=None, help="where to cache the downloaded models")\n    ap.add_argument("--data_dir", default=None, help="folder containing CheXpert-v1.0-small/ (optional: random test images)")\n    ap.add_argument("--host", default="127.0.0.1")\n    ap.add_argument("--port", type=int, default=7860)\n    ap.add_argument("--share", action="store_true", help="public gradio.live link (traffic goes through Gradio\'s servers)")\n    a = ap.parse_args()\n    if not torch.cuda.is_available():\n        raise SystemExit("A CUDA GPU is required (Stable Diffusion sampling in bf16).")\n    root = a.local_root or fetch_assets(a.hf_repo, a.cache_dir, os.environ.get("HF_TOKEN"))\n    E = Engine(root, "cuda", a.data_dir)\n    build_ui(E).queue(default_concurrency_limit=1).launch(server_name=a.host, server_port=a.port, share=a.share)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(work / 'requirements-kaggle.txt').write_text('diffusers==0.42.0\ntransformers==5.20.2\npeft==0.21.1\naccelerate==1.15.0\nsafetensors==0.8.0\nhuggingface-hub==2.3.0\ntorchxrayvision==1.5.7\nscipy==1.17.1\nnumpy==2.4.6\npandas==3.1.0\npillow==13.0.0\ngradio==6.32.0\n', encoding='utf-8')
# Keep Kaggle's working CUDA torch/torchvision pair; isolate other packages.
subprocess.run([sys.executable, "-m", "pip", "--python", demo_python,
                "install", "--upgrade", "-r",
                str(work / "requirements-kaggle.txt")], check=True)
subprocess.run([demo_python, "-c",
    "import torch, torchvision, gradio, diffusers, transformers, peft, torchxrayvision; "
    "assert torch.cuda.is_available(); print('Demo imports and CUDA are ready.')"], check=True)


## 3. Launch the demo and open its link

In [ ]:
import os, secrets, subprocess
from pathlib import Path

work = Path("/kaggle/working/cxr-demo")
demo_python = str(work / ".venv/bin/python")
(work / 'launch.py').write_text('import os\nfrom pathlib import Path\nimport app_kaggle as app\nimport torch\n\nif not torch.cuda.is_available():\n    raise SystemExit("A CUDA GPU is required.")\nprint("Sampling precision:", app.DEMO_DTYPE, flush=True)\nroot = app.fetch_assets(app.HF_REPO, cache_dir=os.environ["HF_HOME"], token=False)\nengine = app.Engine(root, "cuda")\nengine.pipe.vae.enable_slicing()\nui = app.build_ui(engine)\ntry:\n    ui.queue(default_concurrency_limit=1).launch(\n        server_name="127.0.0.1", server_port=7860, share=True,\n        auth=("cxr", os.environ["CXR_DEMO_PASSWORD"]),\n        show_error=True, prevent_thread_lock=False)\nfinally:\n    ui.close()\n', encoding='utf-8')
# Credentials exist only for this run. Keep notebook outputs private.
password = secrets.token_urlsafe(18)
child_env = os.environ.copy()
child_env.update(HF_HOME=str(work / "model-cache"), PYTHONUNBUFFERED="1",
                 CXR_DEMO_PASSWORD=password, GRADIO_ANALYTICS_ENABLED="False")
print("Demo login: cxr")
print("Demo password:", password)
print("Wait for model loading, then open the https://...gradio.live link below.")
print("Leave this cell running while using the demo. Stop it to end GPU use.")
proc = subprocess.Popen([demo_python, "-u", str(work / "launch.py")], cwd=str(work),
                        env=child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
log_path = work / "demo.log"
try:
    with log_path.open("w", encoding="utf-8") as log:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    code = proc.wait()
    if code:
        tail = log_path.read_text(encoding="utf-8")[-12000:]
        raise RuntimeError(f"Demo exited with code {code}. Log: {log_path}\n{tail}")
finally:
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
